In [1]:
import os
import json
import ollama
import chromadb
from typing import List, Dict

# Configuration Constants
JSON_FILE_PATH = "MetadataAPIs.json"
EMBEDDING_MODEL = "nomic-embed-text"
COLLECTION_NAME = "api_metadata_memstore"

# 1. Initialize In-Memory Vector DB Client
chroma_client = chromadb.Client()

# Create or reset collection
try:
    chroma_client.delete_collection(name=COLLECTION_NAME)
except Exception:
    pass
collection = chroma_client.create_collection(name=COLLECTION_NAME)


# 2. Ingest: Load JSON File into Array of API Metadata
def ingest_api_metadata(file_path: str) -> List[Dict]:
    """Loads and validates the API metadata from a JSON file."""
    if not os.path.exists(file_path):
        print(f"Error: {file_path} not found. Please create the file first.")
        return []
        
    with open(file_path, "r", encoding="utf-8") as f:
        try:
            api_array = json.load(f)
            print(f"Successfully ingested {len(api_array)} APIs from {file_path}.")
            return api_array
        except json.JSONDecodeError as e:
            print(f"Failed to parse JSON file: {e}")
            return []


# 3. Embedding Pipeline: Process and Vectorize metadata
def get_embedding(text: str) -> List[float]:
    """Generate vector embedding using local Ollama model."""
    response = ollama.embeddings(model=EMBEDDING_MODEL, prompt=text)
    return response["embedding"]


def index_api_database(api_list: List[Dict]):
    """Iterates through API metadata, builds search contexts, embeds, and stores them."""
    if not api_list:
        print("No metadata available to index.")
        return

    print(f"Generating vectors using '{EMBEDDING_MODEL}'...")
    
    for api in api_list:
        # Build dense text block for semantic analysis
        semantic_text = f"Title: {api['title']}. Description: {api['description']}. Keywords: {' '.join(api['tags'])}"
        
        # Pass to the embedding pipeline
        vector = get_embedding(semantic_text)
        
        # Save to Vector Store
        collection.add(
            documents=[semantic_text],
            metadatas=[{
                "api_id": api["api_id"], 
                "title": api["title"], 
                "category": api["category"]
            }],
            ids=[api["api_id"]],
            embeddings=[vector]
        )
    print("Vector database indexing complete.")


# 4. Semantic Search Interface
def search_api(user_query: str, top_k: int = 1):
    """Searches for the closest API vector match using cosine/distance similarity."""
    print(f"\n[Search Query]: '{user_query}'")
    
    # Vectorize the user's natural language question
    query_vector = get_embedding(user_query)
    
    # Query database
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=top_k
    )
    
    # Display the results neatly
    if results and results["metadatas"] and results["metadatas"][0]:
        for i in range(len(results["metadatas"][0])):
            metadata = results["metadatas"][0][i]
            document = results["documents"][0][i]
            print(f"-> Top Match Found: {metadata['title']} ({metadata['category']})")
            print(f"   Indexed Text: {document}")
    else:
        print("-> No matches found.")


# --- Execution Flow ---
if __name__ == "__main__":
    # Step 1: Run Ingestion
    loaded_apis = ingest_api_metadata(JSON_FILE_PATH)
    
    # Step 2: Feed data to embedding system
    index_api_database(loaded_apis)
    
    # Step 3: Test natural language semantic retrieval
    search_api("How can my application send phone verification codes via SMS?", top_k=1)
    search_api("I need a reliable tool to handle credit card billing cycles.", top_k=1)


Successfully ingested 4 APIs from MetadataAPIs.json.
Generating vectors using 'nomic-embed-text'...
Vector database indexing complete.

[Search Query]: 'How can my application send phone verification codes via SMS?'
-> Top Match Found: Twilio SMS & Voice API (Communication)
   Indexed Text: Title: Twilio SMS & Voice API. Description: Programmatically send text messages, initiate phone calls, and build automated verification systems.. Keywords: sms voice otp phone text

[Search Query]: 'I need a reliable tool to handle credit card billing cycles.'
-> Top Match Found: Stripe Payment Gateway (Finance)
   Indexed Text: Title: Stripe Payment Gateway. Description: Accept global payments, process credit cards, manage recurring billing, and track payouts securely.. Keywords: payment billing credit-card subscription checkout
